# Analytic stellarator fixed points and workflow geometry

This tutorial validates the generic topology workflow against a real field-line integration in pyna's public analytic stellarator model.  The computation has three steps:

1. build a continuous-time field-line flow and its period-4 Poincare return map;
2. find and classify closed O/X fixed points by integrating the flow;
3. promote the numerical fixed points into `PeriodicOrbit` and `IslandChain` geometry objects.

The example uses an analytic model, so it does not require external device data files.

In [ ]:
import numpy as np

from pyna.dynamics import CallableMap
from pyna.topo import TopologyWorkflow
from pyna.topo.fixed_points import classify_fixed_point, find_periodic_orbit, poincare_map
from pyna.toroidal.equilibrium.stellarator import simple_stellarator

The analytic stellarator exposes a 3-D tangent field `(dR/ds, dZ/ds, dphi/ds)`.  A Poincare map on a fixed toroidal section needs the reduced system `d(R,Z)/dphi`, so the small adapter below changes the independent variable from arc length to `phi`.

In [ ]:
def field_func_2d(stellarator):
    def ff2d(R: float, Z: float, phi: float) -> np.ndarray:
        tangent = np.asarray(stellarator.field_func(np.array([R, Z, phi])), dtype=float)
        dphi_ds = tangent[2]
        if abs(dphi_ds) < 1e-15:
            return np.zeros(2)
        return np.array([tangent[0] / dphi_ds, tangent[1] / dphi_ds])

    return ff2d

The model below is tuned to have a visible period-4 island chain near the `q = 5/4` resonance.  `find_periodic_orbit` scans around the resonant radius and refines closed points of the fourth return map.

In [ ]:
st = simple_stellarator(
    R0=3.0,
    r0=0.3,
    B0=1.0,
    q0=1.1,
    q1=5.0,
    m_h=4,
    n_h=4,
    epsilon_h=0.05,
)
ff2d = field_func_2d(st)

psi_res = st.resonant_psi(5, 4)[0]
r_res = st.r0 * np.sqrt(psi_res)
seed = np.array([st.R0 + r_res, 0.0])

points = find_periodic_orbit(
    ff2d,
    seed=seed,
    n_turns=4,
    r_scan=r_res * 0.6,
    n_scan=300,
    verbose=False,
)
len(points)

Classification comes from the monodromy of the return map.  Elliptic points are O-points, while hyperbolic points are X-points.  The residual checks that the numerical point is closed under the period-4 map.

In [ ]:
records = []
for point in points:
    kind, jacobian, det_jac = classify_fixed_point(point, ff2d, n_turns=4)
    residual = float(np.linalg.norm(poincare_map(point, ff2d, n_turns=4) - point))
    records.append(
        {
            "kind": kind,
            "point": np.asarray(point, dtype=float),
            "jacobian": jacobian,
            "det": det_jac,
            "residual": residual,
        }
    )

for index, record in enumerate(records):
    R, Z = record["point"]
    print(
        f"{index}: kind={record['kind']}  R={R:.8f}  Z={Z:.8f}  "
        f"det={record['det']:.6f}  residual={record['residual']:.2e}"
    )

The workflow layer turns numerical arrays into persistent geometry objects.  Downstream code can then reason about the same object shape for maps, field-line systems, Hamiltonian systems, or external fixed-point data.

In [ ]:
o_record = next(record for record in records if record["kind"] == "O")
x_record = next(record for record in records if record["kind"] == "X")

return_map = CallableMap(
    lambda x: poincare_map(x, ff2d, n_turns=4),
    dim=2,
    coordinate_names=("R", "Z"),
    label="analytic stellarator P^4",
)
workflow = TopologyWorkflow(closure_tol=1e-7)

o_orbit = workflow.periodic_orbit(
    [o_record["point"]],
    map_obj=return_map,
    stability_data=o_record["jacobian"],
    coordinate_names=("R", "Z"),
    metadata={"kind": "O"},
)
x_orbit = workflow.periodic_orbit(
    [x_record["point"]],
    map_obj=return_map,
    stability_data=x_record["jacobian"],
    coordinate_names=("R", "Z"),
    metadata={"kind": "X"},
)

chain = workflow.island_chain(
    [o_orbit],
    [x_orbit],
    proximity_tol=1.0,
    label="q=5/4 analytic stellarator",
)

print("islands:", chain.n_islands)
print("O stability:", o_orbit.stability_data.classification.name)
print("X stability:", x_orbit.stability_data.classification.name)
print("O point:", chain.O_points[0].state)
print("X point:", chain.X_points[0].state)

The important point is not that the builder hides the math.  The builder keeps the conversion explicit: integration produces candidate fixed points, classification attaches stability data, and the workflow object gives the rest of pyna a common geometry contract.